# Aromatase Bioactivity Analysis — ChEMBL Dataset (First pandas project)

Author: Ogerovor Joy Ben
Date: July 2026

## Objective
Explore bioactivity data for human aromatase from ChEMBL, filter to IC50-based measurements, clean data quality issues, and classify compounds by potency (Active/Intermediate/Inactive).

## Why aromatase?
Aromatase converts androgens to estrogens and is relevant to PCOS-related hyperandrogenism — part of a broader multi-target phytochemical screening project.

## Data source
ChEMBL database (chembl.ebi.ac.uk), target: Aromatase 
(Homo sapiens*, single protein), downloaded [july 24 2026] as CSV.

## Load data
chEMBL exports are semicolon-delimited despite the .csv extension.

In [2]:
import pandas as pd 
aro = pd.read_csv('/Users/mac/Downloads/aromatase_bioactivity.csv', sep= ';')

In [6]:
aro.head()
aro.info()
aro.describe()
aro.shape
aro['Standard Type'].value_counts()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7040 entries, 0 to 7039
Data columns (total 48 columns):
 #   Column                      Non-Null Count  Dtype  
---  ------                      --------------  -----  
 0   Molecule ChEMBL ID          7040 non-null   object 
 1   Molecule Name               677 non-null    object 
 2   Molecule Max Phase          468 non-null    float64
 3   Molecular Weight            7040 non-null   float64
 4   #RO5 Violations             7011 non-null   float64
 5   AlogP                       7011 non-null   float64
 6   Compound Key                7040 non-null   object 
 7   Smiles                      7040 non-null   object 
 8   Standard Type               7040 non-null   object 
 9   Standard Relation           6752 non-null   object 
 10  Standard Value              6733 non-null   float64
 11  Standard Units              6573 non-null   object 
 12  pChEMBL Value               4477 non-null   float64
 13  Data Validity Comment       270 n

Standard Type
IC50                  4413
Inhibition            1174
Ki                     653
Activity               153
Relative potency       111
Km                      77
Ratio IC50              61
Ratio                   57
pIC50                   47
Imax                    41
Vmax                    37
RP                      29
K inact                 26
Relative Potency        21
Inhibition potency      17
EC50                    14
Activity remaning       12
Log IC50                12
kon                      8
Ks                       8
k_off                    8
Ratio LC50/IC50          7
k cat                    6
deltaG                   5
INH                      4
T1/2                     4
Km/Ki                    4
Ki app                   4
Kinact                   4
Effect                   4
Ks app                   4
kinact                   3
Ka                       3
TDI                      2
Drug metabolism          2
Ka app                   2
ED50          

## Filter to IC50 measurements only
'Standard Type' contains many incompatible measurement types(IC50, Ki,Km,EC50 etc). These measure different things and cant be compared directly, so i isolated IC50- the largest and most standard type in this dataset.

## Remove ChEMBL-flagged invalid entries
Initial summary statistics were heavily distorted by a small number of rows ChEMBL itself flagged as "Outside typical range" in `Data Validity Comment` (e.g. one row reported IC50 ≈ 3.4 × 10¹³ nM). Rather than picking an arbitrary cutoff, we trust ChEMBL's own curation and drop rows with any validity flag.

In [29]:
aro_ic50 = aro[aro['Standard Type']== 'IC50']
aro_ic50.shape
aro_ic50.isnull().sum()
aro_ic50 = aro_ic50[aro_ic50['Data Validity Comment'].isnull()]


## Select relevant columns
Of 48 original columns, only a handful matter for potency and drug-likeness analysis: identifier, structure, molecular weight, Lipinski RO5 violations, and the activity measurement itself.


In [30]:
aro_filtered = aro_ic50[['Molecule ChEMBL ID','Smiles','Molecular Weight','#RO5 Violations','Standard Type','Standard Value','Standard Units']]
aro_filtered.head()

,Molecule ChEMBL ID,Smiles,Molecular Weight,#RO5 Violations,Standard Type,Standard Value,Standard Units
0,CHEMBL1397,CC[C@@H]([C@H](C)O)n1ncn(-c2ccc(N3CCN(c4ccc(OC...,700.79,2.0,IC50,8400.00,nM
4,CHEMBL275594,CCCC1(c2ccncc2)CCC(=O)NC1=O,232.28,0.0,IC50,6000.00,nM
5,CHEMBL168444,O=C1c2ccccc2CCC1C(O)c1ccncn1,254.29,0.0,IC50,27000.00,nM
6,CHEMBL223452,N#Cc1ccc(C(c2ccc(C#N)cc2)c2cncnc2)cc1,296.33,0.0,IC50,288.40,nM
7,CHEMBL375043,N#Cc1ccc(Cn2ccnc2)c2ccccc12,233.27,0.0,IC50,10.96,nM


## Drop rows missing potency values
A small number of rows had no `Standard Value` — without this, a row can't be used in any numeric analysis.


In [31]:
aro_filtered = aro_filtered.dropna(subset= ['Standard Value'])
aro_filtered.shape

(4115, 7)

## Classify compounds by potency
Using a strict Active cutoff (< 100 nM) just for my preference

- Active: IC50 < 100 nM
- Intermediate: 100 nM ≤ IC50 ≤ 1000 nM
- Inactive: IC50 > 1000 nM

In [32]:
def classify_activity(value):
    if value < 100:
        return'Active'
    elif value <= 1000:
        return 'Intermediate'
    else:
        return 'Inactive'

aro_filtered["Bioactivity Class"]= aro_filtered['Standard Value'].apply(classify_activity)
aro_filtered.head()

,Molecule ChEMBL ID,Smiles,Molecular Weight,#RO5 Violations,Standard Type,Standard Value,Standard Units,Bioactivity Class
0,CHEMBL1397,CC[C@@H]([C@H](C)O)n1ncn(-c2ccc(N3CCN(c4ccc(OC...,700.79,2.0,IC50,8400.00,nM,Inactive
4,CHEMBL275594,CCCC1(c2ccncc2)CCC(=O)NC1=O,232.28,0.0,IC50,6000.00,nM,Inactive
5,CHEMBL168444,O=C1c2ccccc2CCC1C(O)c1ccncn1,254.29,0.0,IC50,27000.00,nM,Inactive
6,CHEMBL223452,N#Cc1ccc(C(c2ccc(C#N)cc2)c2cncnc2)cc1,296.33,0.0,IC50,288.40,nM,Intermediate
7,CHEMBL375043,N#Cc1ccc(Cn2ccnc2)c2ccccc12,233.27,0.0,IC50,10.96,nM,Active


## Summary statistics by activity class


In [33]:

aro_filtered.groupby('Bioactivity Class')['Standard Value'].agg(['mean','median','std'])

,mean,median,std
Bioactivity Class,,,
Active,30.035494,21.00,28.347964
Inactive,14039.689831,5785.48,19653.933853
Intermediate,436.764135,349.95,278.894324


In [34]:
aro_filtered['Bioactivity Class'].value_counts()

Bioactivity Class
Inactive        1958
Intermediate    1127
Active          1030
Name: count, dtype: int64

In [24]:
aro_filtered[aro_filtered['Bioactivity Class']== 'Inactive']['Standard Value'].sort_values(ascending=False).head(10)

4653    3.388442e+13
733     1.819701e+13
732     4.677351e+12
2495    2.818383e+12
4535    1.949845e+12
2629    7.762471e+11
5924    5.128614e+11
6505    9.977001e+08
5893    9.749896e+08
6974    9.682779e+08
Name: Standard Value, dtype: float64

In [26]:
aro_ic50.loc[[4653,733,732,2495,4535,2629,5924,6505,5893,6974], ['Standard Value', 'Standard Units', 'Data Validity Comment']]

,Standard Value,Standard Units,Data Validity Comment
4653,3.388442e+13,nM,Outside typical range
733,1.819701e+13,nM,Outside typical range
732,4.677351e+12,nM,Outside typical range
2495,2.818383e+12,nM,Outside typical range
4535,1.949845e+12,nM,Outside typical range
2629,7.762471e+11,nM,Outside typical range
5924,5.128614e+11,nM,Outside typical range
6505,9.977001e+08,nM,Outside typical range
5893,9.749896e+08,nM,Outside typical range
6974,9.682779e+08,nM,Outside typical range


In [35]:
aro_filtered.to_csv('/Users/mac/Downloads/aromatase_ic50_clean_labeled.csv', index=False)

# Aromatase Bioactivity Analysis

## Objective
Explore chEMBL bioactivity data for human aromatase, filter to IC50 measurements, and classsify compounds as Active/Intermediate/inactive based on potency threshold.

## Data Source
chEMBL database, target: Aromatase(Homo sapiens)
Downloaded: 24th July 2026

## Bioactivity classification
- Active: IC50 < 100 nM
- Intermediate: 100 nM <= IC50 <= 1000 nM
- Inactive: IC50 > 1000 nM

## Key cleaning steps
1. Filtered to Standard Type == IC50 only (excluded Ki, Ratio, etc)
2. Selected relevant columns (Smiles, Molecular weight, R05 violations, standard value)
3. Dropped rows with missing standard value
4. Removed rows flagged by chEMBL's Data Validity comment("Outside typical range") -this fixed severely skewed summary statistics.

## Results
Of 4,215 IC50-labeled aromatase compounds (after removing ChEMBL-flagged invalid entries):
- Inactive: 1,958 compounds (mean IC50 ≈ 14,040 nM)
- Intermediate: 1,127 compounds (mean IC50 ≈ 437 nM)
- Active: 1,030 compounds (mean IC50 ≈ 30 nM)

## Data quality note
Initial summary statistics were heavily distorted by a small number of ChEMBL entries flagged "Outside typical range" — one row had a reported IC50 of ~3.4 × 10^13 nM. Removing validity-flagged rows corrected the Inactive class mean from 2.9 × 10^10 nM to a biologically plausible 14,040 nM, while the median barely shifted - a useful reminder to check meaan vs median divergence before trusting summary statistics.


